# Phase 3: Temperature Clustering (FMIC/SMIC)

This notebook adapts the validated Phase 2 workflow for the Phase 3 temperature-only clustering experiment. Stages 1-4 prepare and validate the matched Phase 2 reference baseline. Stage 5 implements US-3.2 history, US-3.3 scalar normalization, and US-3.4 partition quality and greedy merging for the complete alpha sweep. Existing event and row-audit outputs retain the calculations. FMIC/SMIC allowances, coordinator behavior, and clustered monitoring remain later work.


In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

## Stage 1: Input Loading and Policy Resolution

This stage validates input artifacts, resolves preferred and fallback source policies, and normalizes numeric columns required by the deterministic state machine.

Run context note: repository paths resolve from `Path.cwd()` first and fall back to its parent when the notebook is executed from the `scripts` directory. If artifacts still are not found under the resolved project root, run the notebook from the expected repository root or adjust path resolution.

In [ ]:
# Stage 1: Load and normalize Phase 1 inputs for deterministic processing.
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = resolve_repo_root()
PHASE1_ALIGNED_PATH = REPO_ROOT / "artifacts/aligned_phase1_temperature.csv"
PHASE1_STATS_PATH = REPO_ROOT / "artifacts/phase1_average_statistics.csv"
PHASE2_OUTPUT_PATH = REPO_ROOT / "artifacts/phase2_temperature_sensors.csv"
PHASE2_METRICS_PATH = REPO_ROOT / "artifacts/phase2_temperature_metrics.json"

# Strict all-devices policy: use the observed-only all-devices p90 threshold and derive
# the row-level observed global average from the current minute's sensor readings.
PRIMARY_SOURCE_SERIES_NAME = "all_devices_observed"
PRIMARY_GLOBAL_AVG_COLUMN = "average_temperature_all_devices"

SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
GLOBAL_AVG_COLUMN = PRIMARY_GLOBAL_AVG_COLUMN

# Check to see if the paths actually exist
def require_input_files(paths):
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")


# Obtains the primary global average column
def resolve_global_average_column(df):
    # Use the all-device aggregate as the strict global source.
    if PRIMARY_GLOBAL_AVG_COLUMN in df.columns:
        return PRIMARY_GLOBAL_AVG_COLUMN
    raise ValueError(
        "Could not find the required all-devices global average column in aligned phase1 data"
    )


def select_p90_row(stats_frame):
    # Use the all-devices p90 row as the strict threshold source.
    row = stats_frame.loc[stats_frame["series_name"] == PRIMARY_SOURCE_SERIES_NAME]
    if row.empty:
        raise ValueError(
            "Could not find the required all-devices p90 source row in phase1_average_statistics.csv"
        )
    return row


def find_sensor_columns(df):
    # Per-sensor columns follow *_temperature; exclude aggregate averages.
    cols = sorted(
        [
            c
            for c in df.columns
            if c.endswith("_temperature")
            and c not in [PRIMARY_GLOBAL_AVG_COLUMN]
        ]
    )
    if not cols:
        raise ValueError("No per-sensor temperature columns found in aligned phase1 data")
    return cols


require_input_files([PHASE1_ALIGNED_PATH, PHASE1_STATS_PATH])

phase1_df = pd.read_csv(PHASE1_ALIGNED_PATH, keep_default_na=False, na_values=[""])
stats_df = pd.read_csv(PHASE1_STATS_PATH)

if phase1_df.empty:
    raise ValueError("aligned_phase1_temperature.csv has no rows")

GLOBAL_AVG_COLUMN = resolve_global_average_column(phase1_df)
SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
p90_row = select_p90_row(stats_df)

p90_threshold = float(p90_row.iloc[0]["p90"])
sensor_cols = find_sensor_columns(phase1_df)

phase1_df = phase1_df.sort_values("bucket_epoch", kind="mergesort").reset_index(drop=True)
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.phase2_monitoring import normalize_phase2_readings
phase1_df = normalize_phase2_readings(phase1_df, [GLOBAL_AVG_COLUMN, *sensor_cols])

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "rows": int(len(phase1_df)),
            "sensor_count": int(len(sensor_cols)),
            "p90_threshold": p90_threshold,
            "global_avg_column": GLOBAL_AVG_COLUMN,
            "source_series_name": SOURCE_SERIES_NAME,
        },
        indent=2,
    )
)

## Stage 2: Membership-Aware Deterministic State Machine

Online status is known without messages. Every departure or return triggers synchronization.
Startup and each completed sync contact only currently online sensors: N requests, N responses, N threshold updates.
Coinciding causes share one sync; scheduled negative margin has primary precedence. Forced rows suppress local triggers.
Local deviations and allowance violations are still checked against entry state for reporting on forced rows; this does not change traffic or sync causes.
State updates apply on the next row. Zero observations mean undefined global state, pending synchronization, and zero traffic.
The testable state-machine implementation is in scripts/phase2_monitoring.py; no clustering is implemented here.


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.phase2_monitoring import build_phase2_trace, check_membership_invariants, check_trace_invariants

phase2_df = build_phase2_trace(phase1_df, sensor_cols, p90_threshold)
trace_errors = check_membership_invariants(phase2_df)
if trace_errors:
    raise ValueError("Phase 2 membership validation failed: " + "; ".join(trace_errors))

sensor_count = len(sensor_cols)
public_phase2_df = phase2_df.drop(
    columns=["event_resync_consumed_from_prior_row", "event_resync_triggered_by_local_violation"]
)
PHASE2_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
public_phase2_df.to_csv(PHASE2_OUTPUT_PATH, index=False)
print(f"Wrote {len(phase2_df)} rows to {PHASE2_OUTPUT_PATH}")


## Stage 3: Metrics and Communication Diagnostics

This stage derives classification-style diagnostics, communication cost comparisons, and detection delay summaries from the generated Phase 2 dataset.

Interpretation note: lag values are bucket offsets from the start of each contiguous violation window, not absolute row indices or timestamps.

In [ ]:
def compute_confusion_counts(actual_series, predicted_series):
    tp = int(((actual_series == 1) & (predicted_series == 1)).sum())
    fn = int(((actual_series == 1) & (predicted_series == 0)).sum())
    fp = int(((actual_series == 0) & (predicted_series == 1)).sum())
    tn = int(((actual_series == 0) & (predicted_series == 0)).sum())
    return tp, fn, fp, tn

def compute_window_lags(actual_series, predicted_series):
    # Lag unit is buckets since the start of each contiguous actual-positive window.
    lags = []
    in_window = False
    window_start = None

    for idx, val in enumerate(actual_series.tolist()):
        if val == 1 and not in_window:
            in_window = True
            window_start = idx
        elif val != 1 and in_window:
            window_end = idx - 1
            segment = predicted_series.iloc[window_start : window_end + 1]
            hit_positions = np.where(segment.to_numpy() == 1)[0]
            if len(hit_positions) > 0:
                lags.append(int(hit_positions[0]))
            else:
                lags.append(None)
            in_window = False

    if in_window and window_start is not None:
        segment = predicted_series.iloc[window_start:]
        hit_positions = np.where(segment.to_numpy() == 1)[0]
        if len(hit_positions) > 0:
            lags.append(int(hit_positions[0]))
        else:
            lags.append(None)

    return lags

def round4(value):
    return round(float(value), 4)

def compute_row_violation_detection_delay(actual_series, predicted_series):
    # For each violating row, emit the first-hit lag within its contiguous violation window.
    row_delay = [None] * len(actual_series)
    actual_list = actual_series.tolist()
    predicted_list = predicted_series.tolist()

    in_window = False
    window_start = None

    for idx, val in enumerate(actual_list):
        if val == 1 and not in_window:
            in_window = True
            window_start = idx
        elif val != 1 and in_window:
            window_end = idx - 1
            segment = predicted_list[window_start : window_end + 1]
            hit_positions = np.where(np.array(segment) == 1)[0]
            lag_value = int(hit_positions[0]) if len(hit_positions) > 0 else None
            for window_idx in range(window_start, window_end + 1):
                row_delay[window_idx] = lag_value
            in_window = False

    if in_window and window_start is not None:
        segment = predicted_list[window_start:]
        hit_positions = np.where(np.array(segment) == 1)[0]
        lag_value = int(hit_positions[0]) if len(hit_positions) > 0 else None
        for window_idx in range(window_start, len(actual_list)):
            row_delay[window_idx] = lag_value

    return row_delay

actual = phase2_df["global_violation"]
predicted = phase2_df["event_resync_performed"].astype(int)

tp, fn, fp, tn = compute_confusion_counts(actual, predicted)

recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

sensor_count = len(sensor_cols)
baseline_sensor_messages = len(phase2_df) * sensor_count
observed_baseline_messages = int(phase2_df["observed_online_sensor_count"].sum())

distributed_total_messages = int(phase2_df["total_message_count"].sum())
distributed_trigger_messages = int(phase2_df["trigger_message_count"].sum())
distributed_request_messages = int(phase2_df["request_message_count"].sum())
distributed_response_messages = int(phase2_df["response_message_count"].sum())
distributed_broadcast_messages = int(phase2_df["broadcast_message_count"].sum())

# Primary ratio uses total distributed traffic; legacy ratio preserves old trigger-only baseline comparison.
communication_reduction_ratio_primary = (
    1.0 - (distributed_total_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)
communication_reduction_ratio_legacy = (
    1.0 - (distributed_trigger_messages / baseline_sensor_messages)
    if baseline_sensor_messages > 0
    else np.nan
)
observed_reduction_ratio = (
    1.0 - distributed_total_messages / observed_baseline_messages
    if observed_baseline_messages > 0 else np.nan
)

window_lags = compute_window_lags(actual, predicted)
valid_lags = [x for x in window_lags if x is not None]
row_detection_delay = compute_row_violation_detection_delay(actual, predicted)
phase2_df["violation_detection_delay_buckets"] = row_detection_delay

false_safe_count = int(phase2_df["false_safe"].sum())
false_alert_count = int(phase2_df["false_alert"].sum())

# Metrics sections document rule definitions, quality diagnostics, and communication cost.
metrics = {
    "rows": int(len(phase2_df)),
    "sensor_count": int(sensor_count),
    "evaluable_rows": int(actual.notna().sum()),
    "undefined_global_rows": int(actual.isna().sum()),
    "membership_policy": {
        "mode": "dynamic_observed_only",
        "departure": "first_missing_bucket_after_phase1_freshness_filtering",
        "return": "first_subsequent_valid_reading",
        "availability_detection_messages": 0,
        "sync_recipients": "currently_online_sensors_only",
        "startup": "count_initial_synchronization",
        "zero_observations": "undefined_global_state_pending_sync_zero_traffic",
        "cause_attribution": "overlapping_cause_views_do_not_add_to_transmission_totals",
    },
    "synchronization_policy": {
        "initialization": "first_bucket_with_valid_observations_and_counted_traffic",
        "triggers": "local_allowance_violation_membership_change_or_pending_negative_margin",
        "multiple_causes": "one_combined_synchronization_per_bucket",
        "online_fanout": "one_request_response_and_threshold_update_per_online_sensor",
        "offline_sensor_traffic": 0,
        "state_activation": "next_row",
    },
    "p90_threshold": round4(p90_threshold),
    "p90_threshold_full_precision": float(p90_threshold),
    "threshold_source": {
        "artifact": PHASE1_STATS_PATH.name,
        "series_name": "all_devices_observed",
        "column": "p90",
    },
    "boundary_policy": {
        "canonical_rule": "violated_if_observed_global_average_gte_p90",
        "variant_policy": "strict_gt_available_for_diagnostic_sensitivity_only",
    },
    "trigger_policy": {
        "local_trigger_rule": "event_{sensor}_local_deviation >= entry_{sensor}_local_margin",
        "predicted_positive_rule": "event_resync_performed == 1",
        "diagnostic_interpretation": "Global threshold versus same-row synchronization, not actual warning accuracy; initialization and membership syncs can count as false alerts.",
        "distributed_constraint_state_rule": "Any valid entry local-allowance violation, including reporting-only comparisons on forced rows; FEASIBLE requires all online sensors comparable; otherwise NOT_EVALUABLE, or UNDEFINED with no observations.",
    },
    "confusion": {
        "tp": tp,
        "fn": fn,
        "fp": fp,
        "tn": tn,
    },
    "false_negative_count": fn,
    "false_positive_count": fp,
    "false_safe_count": false_safe_count,
    "false_alert_count": false_alert_count,
    "recall_diagnostic": None if np.isnan(recall) else round4(recall),
    "precision_diagnostic": None if np.isnan(precision) else round4(precision),
    "communication": {
        "headline_reduction_metric": "communication_reduction_ratio_observed",
        "headline_baseline_definition": "one_transmission_per_observed_sensor_per_bucket",
        "distributed_total_messages": distributed_total_messages,
        "distributed_trigger_messages": distributed_trigger_messages,
        "distributed_request_messages": distributed_request_messages,
        "distributed_response_messages": distributed_response_messages,
        "distributed_broadcast_messages": distributed_broadcast_messages,
        "centralized_sensor_messages": int(baseline_sensor_messages),
        "centralized_baseline_definition": "all_configured_sensors_every_bucket_counterfactual",
        "centralized_observed_sensor_messages": observed_baseline_messages,
        "communication_reduction_ratio_observed": None
        if np.isnan(observed_reduction_ratio) else round4(observed_reduction_ratio),
        "membership_resync_messages": int(phase2_df["membership_resync_message_count"].sum()),
        "initialization_messages": int(phase2_df["initialization_message_count"].sum()),
        "communication_reduction_ratio_primary": None
        if np.isnan(communication_reduction_ratio_primary)
        else round4(communication_reduction_ratio_primary),
        "communication_reduction_ratio_legacy": None
        if np.isnan(communication_reduction_ratio_legacy)
        else round4(communication_reduction_ratio_legacy),
    },
    "resync_events": {
        "resync_performed_count": int(phase2_df["event_resync_performed"].sum()),
        "negative_delta_detected_count": int(phase2_df["event_exit_delta_negative"].sum()),
        "scheduled_from_prior_count": int(phase2_df["event_resync_consumed_from_prior_row"].sum()),
        "initialization_count": int(phase2_df["event_initialization_performed"].sum()),
        "membership_change_count": int(phase2_df["event_membership_changed"].sum()),
        "membership_resync_count": int(phase2_df["event_resync_triggered_by_membership_change"].sum()),
        "pending_bucket_count": int(phase2_df["exit_resync_pending"].sum()),
    },
    "detection_delay_by_window": {
        "window_lags_in_buckets": window_lags,
        "mean_lag": None if not valid_lags else round4(np.mean(valid_lags)),
        "max_lag": None if not valid_lags else int(max(valid_lags)),
    },
}

with PHASE2_METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# Rewrite row output so row-level detection delay and observability fields are persisted.
phase2_df.to_csv(PHASE2_OUTPUT_PATH, index=False)

PHASE2_DICTIONARY_PATH = REPO_ROOT / "artifacts/phase2_data_dictionary.csv"
PHASE2_ANALYSIS_PATH = REPO_ROOT / "artifacts/phase2_sensor_reduction_analysis.csv"

def _detect_sensors(columns):
    sensors = set()
    pattern = re.compile(r"^observed_(.+)_temperature$")
    for column in columns:
        match = pattern.match(column)
        if match:
            sensors.add(match.group(1))
    return sorted(sensors)

def _describe_column(column, sensor_names):
    membership_fields = {
        "observed_online_sensor_names": ("membership", "Pipe-delimited IDs with valid readings in the current bucket.", "text"),
        "observed_online_sensor_count": ("membership", "Count of currently online sensors; denominator of observed_global_average.", "count"),
        "event_departing_sensor_names": ("membership", "Previous online set minus current online set; empty on the first row.", "text"),
        "event_returning_sensor_names": ("membership", "Current online set minus previous online set; empty on the first row.", "text"),
        "event_membership_changed": ("membership", "1 when online membership differs from the preceding row; first row is initialization.", "binary (0/1)"),
        "global_state_evaluable": ("diagnostic", "1 when observed_online_sensor_count > 0; otherwise global labels are undefined.", "binary (0/1)"),
        "event_resync_triggered_by_membership_change": ("event", "1 when a completed synchronization includes a membership-change cause, including a deferred change.", "binary (0/1)"),
        "event_initialization_performed": ("event", "1 on the first completed synchronization with valid observations; initialization traffic is counted.", "binary (0/1)"),
        "event_resync_causes": ("event", "Pipe-delimited causes, including deferred causes; negative-margin cause has primary precedence.", "text"),
        "exit_resync_pending": ("state", "1 when synchronization remains due after this row; zero-online buckets cannot complete it.", "binary (0/1)"),
        "exit_resync_pending_causes": ("state", "Pipe-delimited causes awaiting the next usable bucket, including scheduled negative margin.", "text"),
        "membership_resync_message_count": ("communication_attribution", "Request/response/update traffic for a sync including membership change; overlaps causes, never added to total.", "messages"),
        "initialization_message_count": ("communication_attribution", "Request/response/update traffic for initialization; overlaps causes, never added to total.", "messages"),
    }
    if column in membership_fields:
        category, calculation, units = membership_fields[column]
        return {
            "field_name": column, "category": category, "measures": calculation,
            "calculation": calculation, "units": units,
        }
    static = {
        "bucket_epoch": {
            "category": "time",
            "measures": "Unix epoch second for the bucket row.",
            "calculation": "Copied from aligned Phase 1 bucket timestamp.",
            "units": "seconds since 1970-01-01 UTC",
        },
        "bucket_time_utc": {
            "category": "time",
            "measures": "UTC timestamp for the bucket row.",
            "calculation": "Formatted datetime representation of bucket_epoch.",
            "units": "UTC datetime",
        },
        "transition_summary": {
            "category": "state",
            "measures": "Compact state transition label for row auditability.",
            "calculation": "Composed from row event conditions in the state machine.",
            "units": "text",
        },
        "p90_threshold_used": {
            "category": "threshold",
            "measures": "Global P90 threshold used for this run.",
            "calculation": "Loaded from phase1_average_statistics.csv for the observed-only series all_devices_observed.",
            "units": "temperature",
        },
        "observed_global_average": {
            "category": "global",
            "measures": "Current row global average temperature.",
            "calculation": "Mean of currently online valid Phase 1 sensor readings; undefined when count is zero.",
            "units": "temperature",
        },
        "entry_xbar_t0": {
            "category": "global",
            "measures": "Global reference average entering this row.",
            "calculation": "Prior synchronized global reference carried into this row.",
            "units": "temperature",
        },
        "entry_delta_global": {
            "category": "global",
            "measures": "Global margin entering this row.",
            "calculation": "p90_threshold_used - entry_xbar_t0.",
            "units": "temperature",
        },
        "exit_delta_global": {
            "category": "global",
            "measures": "Global margin after row event processing.",
            "calculation": "p90_threshold_used - exit global reference (post-event state).",
            "units": "temperature",
        },
        "global_violation": {
            "category": "diagnostic",
            "measures": "Whether row average violates canonical threshold boundary.",
            "calculation": "1 when observed_global_average >= p90_threshold_used, 0 below, blank when undefined.",
            "units": "binary (0/1)",
        },
        "event_triggering_sensor_names": {
            "category": "event",
            "measures": "Sensor names that requested re-sync on this row.",
            "calculation": "Pipe-delimited list of sensors with event_{sensor}_resync_requested == 1.",
            "units": "text",
        },
        "event_resync_request_count": {
            "category": "event",
            "measures": "Count of sensor-local re-sync requests on this row.",
            "calculation": "Sum of event_{sensor}_resync_requested over all sensors.",
            "units": "count",
        },
        "event_any_sensor_requested_resync": {
            "category": "event",
            "measures": "Whether any sensor requested re-sync on this row.",
            "calculation": "1 when event_resync_request_count > 0, else 0.",
            "units": "binary (0/1)",
        },
        "event_resync_performed": {
            "category": "event",
            "measures": "Whether a re-sync event was executed on this row.",
            "calculation": "1 on completed initialization, membership, scheduled-negative-margin, or local-trigger synchronization.",
            "units": "binary (0/1)",
        },
        "event_resync_reason": {
            "category": "event",
            "measures": "Reason code for performed re-sync.",
            "calculation": "Categorical label from row event evaluation.",
            "units": "text",
        },
        "event_exit_delta_negative": {
            "category": "event",
            "measures": "Whether the row exit global margin is negative.",
            "calculation": "1 when exit_delta_global < 0, else 0.",
            "units": "binary (0/1)",
        },
        "exit_resync_scheduled_next_row": {
            "category": "state",
            "measures": "Whether next row must consume a forced re-sync.",
            "calculation": "1 when a negative-margin sync is due or an observation-dependent sync remains pending.",
            "units": "binary (0/1)",
        },
        "exit_xbar_t0_if_resync": {
            "category": "state",
            "measures": "Global reference that becomes next row entry_xbar_t0 after re-sync.",
            "calculation": "Set to observed_global_average when re-sync is performed; otherwise carry prior.",
            "units": "temperature",
        },
        "exit_delta_global_if_resync": {
            "category": "state",
            "measures": "Global margin that becomes next row entry_delta_global after re-sync.",
            "calculation": "Set to p90_threshold_used - exit_xbar_t0_if_resync.",
            "units": "temperature",
        },
        "trigger_message_count": {
            "category": "communication",
            "measures": "Number of sensor trigger messages emitted this row.",
            "calculation": "On local-trigger rows equals event_resync_request_count; forced rows fixed to 0.",
            "units": "messages",
        },
        "request_message_count": {
            "category": "communication",
            "measures": "Number of distributed request messages this row.",
            "calculation": "observed_online_sensor_count when synchronization completes, otherwise 0; offline sensors receive no requests.",
            "units": "messages",
        },
        "response_message_count": {
            "category": "communication",
            "measures": "Number of distributed response messages this row.",
            "calculation": "observed_online_sensor_count when synchronization completes, otherwise 0; offline sensors send no responses.",
            "units": "messages",
        },
        "broadcast_message_count": {
            "category": "communication",
            "measures": "Number of distributed broadcast messages this row.",
            "calculation": "observed_online_sensor_count when synchronization completes, otherwise 0; threshold updates go only to online sensors.",
            "units": "messages",
        },
        "total_message_count": {
            "category": "communication",
            "measures": "Total communication messages for this row.",
            "calculation": "trigger_message_count + request_message_count + response_message_count + broadcast_message_count.",
            "units": "messages",
        },
        "centralized_constraint_state": {
            "category": "correctness",
            "measures": "Centralized oracle threshold state for this row.",
            "calculation": "UNDEFINED with zero observations; otherwise VIOLATING when global_violation == 1, else FEASIBLE.",
            "units": "categorical",
        },
        "distributed_constraint_state": {
            "category": "correctness",
            "measures": "Sensor-local constraint state, independent of synchronization and the global threshold label.",
            "calculation": "VIOLATING if any online sensor deviation >= its entry local allowance; FEASIBLE only if all online sensors are comparable and none violates; NOT_EVALUABLE for incomplete comparisons; UNDEFINED with no readings. Forced rows compare for reporting only and emit no local triggers.",
            "units": "categorical",
        },
        "false_safe": {
            "category": "correctness",
            "measures": "Rows where centralized state violates and no distributed re-sync was implemented on that row.",
            "calculation": "1 when global_violation == 1 and event_resync_performed == 0; blank when global state is undefined.",
            "units": "binary (0/1)",
        },
        "false_alert": {
            "category": "correctness",
            "measures": "Rows where a distributed re-sync was implemented while centralized state is feasible.",
            "calculation": "1 when global_violation == 0 and event_resync_performed == 1; blank when global state is undefined.",
            "units": "binary (0/1)",
        },
        "violation_detection_delay_buckets": {
            "category": "correctness",
            "measures": "Per-row first synchronization lag within a contiguous global-violation window, not local-warning latency.",
            "calculation": "Window-level first-hit lag projected onto rows belonging to that violation window.",
            "units": "buckets",
        },
    }

    if column in static:
        return {
            "field_name": column,
            "category": static[column]["category"],
            "measures": static[column]["measures"],
            "calculation": static[column]["calculation"],
            "units": static[column]["units"],
        }

    sensor_observed = re.match(r"^observed_(.+)_temperature$", column)
    if sensor_observed:
        sensor = sensor_observed.group(1)
        if sensor in sensor_names:
            return {
                "field_name": column,
                "category": "sensor_observed",
                "measures": f"Observed temperature for sensor {sensor} on this row.",
                "calculation": "Copied from the same bucket's observed Phase 1 value; missing readings stay blank.",
                "units": "temperature",
            }

    sensor_reference = re.match(r"^entry_(.+)_reference_value$", column)
    if sensor_reference:
        sensor = sensor_reference.group(1)
        return {
            "field_name": column,
            "category": "sensor_state",
            "measures": f"Entry synchronized reference value for sensor {sensor}.",
            "calculation": "Sensor value stored at most recent synchronization event.",
            "units": "temperature",
        }

    sensor_margin = re.match(r"^entry_(.+)_local_margin$", column)
    if sensor_margin:
        sensor = sensor_margin.group(1)
        return {
            "field_name": column,
            "category": "sensor_state",
            "measures": f"Entry local trigger margin for sensor {sensor}.",
            "calculation": "Shared global margin at the most recent synchronization including this sensor; blank for sensors excluded from that synchronization.",
            "units": "temperature",
        }

    sensor_deviation = re.match(r"^event_(.+)_local_deviation$", column)
    if sensor_deviation:
        sensor = sensor_deviation.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Row local deviation for sensor {sensor}, including reporting-only checks on forced-sync rows.",
            "calculation": f"observed_{sensor}_temperature - entry_{sensor}_reference_value.",
            "units": "temperature",
        }

    sensor_margin_score = re.match(r"^event_(.+)_margin_proximity_score$", column)
    if sensor_margin_score:
        sensor = sensor_margin_score.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Row margin proximity score for sensor {sensor} (positive=acceptable, negative=violating).",
            "calculation": f"entry_delta_global - event_{sensor}_local_deviation.",
            "units": "temperature",
        }

    sensor_request = re.match(r"^event_(.+)_resync_requested$", column)
    if sensor_request:
        sensor = sensor_request.group(1)
        return {
            "field_name": column,
            "category": "sensor_event",
            "measures": f"Whether sensor {sensor} requested re-sync on this row.",
            "calculation": f"1 when event_{sensor}_local_deviation >= entry_{sensor}_local_margin and row is not forced; else 0.",
            "units": "binary (0/1)",
        }

    return {
        "field_name": column,
        "category": "other",
        "measures": "Field included in Phase 2 output schema.",
        "calculation": "See scripts/README.md and notebook logic for detailed derivation.",
        "units": "n/a",
    }

def _build_data_dictionary(raw_columns, sensor_names):
    sensor_set = set(sensor_names)
    rows = [_describe_column(column, sensor_set) for column in raw_columns]
    return pd.DataFrame(rows)

def _as_number(value, field_name):
    if isinstance(value, (int, float)):
        return float(value)
    raise ValueError(f"Expected numeric value for '{field_name}', received: {value!r}")

def _build_reduction_analysis(metrics_dict):
    communication = metrics_dict.get("communication")
    if not isinstance(communication, dict):
        raise ValueError("Missing 'communication' object in metrics JSON.")

    rows = _as_number(metrics_dict.get("rows"), "rows")
    sensor_count = _as_number(metrics_dict.get("sensor_count"), "sensor_count")
    baseline = rows * sensor_count

    actual = _as_number(
        communication.get("distributed_total_messages"), "distributed_total_messages"
    )
    trigger_messages = _as_number(
        communication.get("distributed_trigger_messages"), "distributed_trigger_messages"
    )
    request_messages = _as_number(
        communication.get("distributed_request_messages"), "distributed_request_messages"
    )
    response_messages = _as_number(
        communication.get("distributed_response_messages"), "distributed_response_messages"
    )
    broadcast_messages = _as_number(
        communication.get("distributed_broadcast_messages"), "distributed_broadcast_messages"
    )

    if baseline <= 0:
        raise ValueError("Baseline communications must be > 0.")

    difference = baseline - actual
    ratio = 1.0 - (actual / baseline)
    percent = round(ratio * 100.0, 2)

    false_safe_count = metrics_dict.get("false_safe_count")
    false_alert_count = metrics_dict.get("false_alert_count")

    analysis_rows = [
        {
            "metric": "baseline_every_minute_messages",
            "value": int(round(baseline)),
            "units": "messages",
            "calculation": "rows * sensor_count",
            "notes": "Configured-sensor counterfactual, not the observed-availability headline baseline.",
        },
        {
            "metric": "poc_actual_messages",
            "value": int(round(actual)),
            "units": "messages",
            "calculation": "distributed_total_messages",
            "notes": "Actual total distributed traffic under this Phase 2 implementation.",
        },
        {
            "metric": "reduction_difference_messages",
            "value": int(round(difference)),
            "units": "messages",
            "calculation": "baseline_every_minute_messages - poc_actual_messages",
            "notes": "Reduction against the configured-sensor counterfactual, not the observed-availability headline.",
        },
        {
            "metric": "reduction_percent_primary",
            "value": f"{percent:.2f}",
            "units": "percent",
            "calculation": "reduction_ratio_primary * 100",
            "notes": "Compatibility field: configured-sensor counterfactual percentage, not the headline reduction.",
        },
        {
            "metric": "distributed_trigger_messages",
            "value": int(round(trigger_messages)),
            "units": "messages",
            "calculation": "sum(trigger_message_count)",
            "notes": "Trigger-category communication breakdown.",
        },
        {
            "metric": "distributed_request_messages",
            "value": int(round(request_messages)),
            "units": "messages",
            "calculation": "sum(request_message_count)",
            "notes": "Request-category communication breakdown.",
        },
        {
            "metric": "distributed_response_messages",
            "value": int(round(response_messages)),
            "units": "messages",
            "calculation": "sum(response_message_count)",
            "notes": "Response-category communication breakdown.",
        },
        {
            "metric": "distributed_broadcast_messages",
            "value": int(round(broadcast_messages)),
            "units": "messages",
            "calculation": "sum(broadcast_message_count)",
            "notes": "Broadcast-category communication breakdown.",
        },
        {
            "metric": "rows",
            "value": int(round(rows)),
            "units": "count",
            "calculation": "metrics.rows",
            "notes": "Number of Phase 2 minute buckets.",
        },
        {
            "metric": "sensor_count",
            "value": int(round(sensor_count)),
            "units": "count",
            "calculation": "metrics.sensor_count",
            "notes": "Number of sensors included in Phase 2 run.",
        },
    ]
    observed_baseline = _as_number(
        communication["centralized_observed_sensor_messages"], "centralized_observed_sensor_messages"
    )
    observed_ratio = communication["communication_reduction_ratio_observed"]
    analysis_rows.extend([
        {
            "metric": "baseline_observed_sensor_messages",
            "value": int(observed_baseline),
            "units": "messages",
            "calculation": "sum(observed_online_sensor_count)",
            "notes": "Headline observed-availability baseline; no offline sensor traffic.",
        },
        {
            "metric": "reduction_ratio_observed",
            "value": observed_ratio,
            "units": "ratio",
            "calculation": "1 - poc_actual_messages / baseline_observed_sensor_messages",
            "notes": "Headline reduction ratio; undefined if there are no observed readings.",
        },
        {
            "metric": "reduction_difference_observed_messages",
            "value": int(observed_baseline - actual),
            "units": "messages",
            "calculation": "baseline_observed_sensor_messages - poc_actual_messages",
            "notes": "Matched availability comparison, including startup and membership synchronization.",
        },
    ])
    analysis_rows.insert(0, {
        "metric": "reduction_percent_observed",
        "value": None if observed_baseline == 0 else f"{100 * (1 - actual / observed_baseline):.2f}",
        "units": "percent",
        "calculation": "100 * (1 - poc_actual_messages / baseline_observed_sensor_messages)",
        "notes": "Headline communication reduction against observed availability; undefined with no readings.",
    })
    for field in ("evaluable_rows", "undefined_global_rows"):
        analysis_rows.append({
            "metric": field, "value": metrics_dict[field], "units": "count",
            "calculation": f"metrics.{field}",
            "notes": "Only evaluable rows enter classification diagnostics.",
        })
    for field in ("membership_resync_messages", "initialization_messages"):
        analysis_rows.append({
            "metric": field, "value": communication[field], "units": "messages",
            "calculation": f"communication.{field}",
            "notes": "Cause attribution only; may overlap other causes and is not added to message-type totals.",
        })

    resync = metrics_dict.get("resync_events")
    if isinstance(resync, dict) and isinstance(
        resync.get("resync_performed_count"), (int, float)
    ):
        analysis_rows.append(
            {
                "metric": "resync_performed_count",
                "value": int(round(float(resync["resync_performed_count"]))),
                "units": "count",
                "calculation": "sum(event_resync_performed)",
                "notes": "Count of synchronization events in the run.",
            }
        )

    if isinstance(false_safe_count, (int, float)):
        analysis_rows.append(
            {
                "metric": "false_safe_count",
                "value": int(round(float(false_safe_count))),
                "units": "count",
                "calculation": "sum(false_safe)",
                "notes": "Legacy synchronization diagnostic: globally violating rows without same-row sync, not actual warning accuracy.",
            }
        )

    if isinstance(false_alert_count, (int, float)):
        analysis_rows.append(
            {
                "metric": "false_alert_count",
                "value": int(round(float(false_alert_count))),
                "units": "count",
                "calculation": "sum(false_alert)",
                "notes": "Legacy synchronization diagnostic: globally safe rows with same-row sync, including startup/membership activity; not incorrect global warnings.",
            }
        )

    reduction_analysis = pd.DataFrame(analysis_rows)
    if "bucket_epoch" in reduction_analysis.columns:
        reduction_analysis = reduction_analysis.drop(columns=["bucket_epoch"])
    return reduction_analysis

def _write_csv_atomic(df, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")
    rounded_df = df.copy()
    for column in rounded_df.columns:
        if rounded_df[column].dtype.kind in "ifc":
            rounded_df[column] = rounded_df[column].round(4)
    rounded_df.to_csv(tmp_path, index=False)
    try:
        tmp_path.replace(output_path)
    except PermissionError:
        rounded_df.to_csv(output_path, index=False)
        if tmp_path.exists():
            tmp_path.unlink()

def _export_phase2_report_csvs_from_artifacts(rows_input, metrics_input, data_dictionary_output, analysis_output):
    if not rows_input.exists():
        raise FileNotFoundError(f"Missing rows input CSV: {rows_input}")
    if not metrics_input.exists():
        raise FileNotFoundError(f"Missing metrics input JSON: {metrics_input}")

    raw_rows = pd.read_csv(rows_input)
    metrics_dict = json.loads(metrics_input.read_text(encoding="utf-8"))

    sensor_names = _detect_sensors(list(raw_rows.columns))
    data_dictionary = _build_data_dictionary(list(raw_rows.columns), sensor_names)
    reduction_analysis = _build_reduction_analysis(metrics_dict)

    _write_csv_atomic(data_dictionary, data_dictionary_output)
    _write_csv_atomic(reduction_analysis, analysis_output)

    return {
        "data_dictionary": data_dictionary_output,
        "sensor_reduction_analysis": analysis_output,
    }

report_outputs = _export_phase2_report_csvs_from_artifacts(
    rows_input=PHASE2_OUTPUT_PATH,
    metrics_input=PHASE2_METRICS_PATH,
    data_dictionary_output=PHASE2_DICTIONARY_PATH,
    analysis_output=PHASE2_ANALYSIS_PATH,
)

print(json.dumps(metrics, indent=2))
print(f"Wrote metrics to {PHASE2_METRICS_PATH}")
print("Wrote CSV report outputs:")
for key, value in report_outputs.items():
    print(f"- {key}: {value}")

In [ ]:
INTERNAL_ONLY_COLUMNS = [
    "event_resync_consumed_from_prior_row",
    "event_resync_triggered_by_local_violation",
]

public_rows_df = phase2_df.drop(columns=INTERNAL_ONLY_COLUMNS, errors="ignore")
public_rows_df.to_csv(PHASE2_OUTPUT_PATH, index=False)

report_outputs = _export_phase2_report_csvs_from_artifacts(
    rows_input=PHASE2_OUTPUT_PATH,
    metrics_input=PHASE2_METRICS_PATH,
    data_dictionary_output=PHASE2_DICTIONARY_PATH,
    analysis_output=PHASE2_ANALYSIS_PATH,
)

print("Re-applied public row projection before final validation.")
print("Refreshed CSV report outputs from public row schema:")
for key, value in report_outputs.items():
    print(f"- {key}: {value}")

## Stage 4: Integrated Validation Checks

This stage consolidates smoke checks and export-policy checks directly into notebook execution.

It validates:
- required input/output artifact presence,
- required Phase 2 schema families and column ordering,
- numeric rounding policy,
- required metrics policy sections,
- required report artifacts and report-schema invariants.

Both report artifacts are required in this integrated flow:
- `artifacts/phase2_data_dictionary.csv`
- `artifacts/phase2_sensor_reduction_analysis.csv`

In [ ]:
import re


def _has_more_than_four_decimals(value):
    if pd.isna(value):
        return False
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return False
    return abs(numeric_value - round(numeric_value, 4)) > 1e-12


def _detect_sensor_names_from_rows(rows_df):
    names = set()
    pattern = re.compile(r"^observed_(.+)_temperature$")
    for column in rows_df.columns:
        match = pattern.match(column)
        if match:
            names.add(match.group(1))
    return sorted(names)


def _check_phase2_schema(rows_df, analysis_df):
    errors = []

    if "exit_delta_global" in rows_df.columns:
        errors.append("phase2_temperature_sensors.csv still includes exit_delta_global")

    disallowed_columns = {
        "event_resync_consumed_from_prior_row",
        "event_resync_triggered_by_local_violation",
    }
    leaked_internal_columns = sorted(disallowed_columns.intersection(set(rows_df.columns)))
    if leaked_internal_columns:
        errors.append(
            "phase2_temperature_sensors.csv includes internal-only columns: "
            + ", ".join(leaked_internal_columns)
        )

    columns = list(rows_df.columns)
    required_columns = [
        "entry_xbar_t0",
        "exit_xbar_t0_if_resync",
        "entry_delta_global",
        "exit_delta_global_if_resync",
        "observed_online_sensor_count",
        "event_membership_changed",
        "event_resync_causes",
        "exit_resync_pending",
        "global_state_evaluable",
    ]
    for column in required_columns:
        if column not in columns:
            errors.append(f"Missing expected schema column: {column}")

    if "entry_xbar_t0" in columns and "exit_xbar_t0_if_resync" in columns:
        entry_index = columns.index("entry_xbar_t0")
        exit_index = columns.index("exit_xbar_t0_if_resync")
        if exit_index != entry_index + 1:
            errors.append("exit_xbar_t0_if_resync is not immediately after entry_xbar_t0")

    if "entry_delta_global" in columns and "exit_delta_global_if_resync" in columns:
        entry_index = columns.index("entry_delta_global")
        exit_index = columns.index("exit_delta_global_if_resync")
        if exit_index != entry_index + 1:
            errors.append("exit_delta_global_if_resync is not immediately after entry_delta_global")

    local_margin_columns = [
        column for column in columns if re.match(r"^entry_.+_local_margin$", column)
    ]
    margin_score_columns = [
        column for column in columns if re.match(r"^event_.+_margin_proximity_score$", column)
    ]

    print("phase2_local_margin_cols", len(local_margin_columns))
    print("phase2_margin_proximity_cols", len(margin_score_columns))

    if not local_margin_columns:
        errors.append("Missing expected local margin columns in phase2_temperature_sensors.csv")
    if not margin_score_columns:
        errors.append("Missing expected margin proximity score columns in phase2_temperature_sensors.csv")

    if "bucket_epoch" in analysis_df.columns:
        errors.append("phase2_sensor_reduction_analysis.csv still includes bucket_epoch")

    return errors


def _check_phase2_rounding(rows_df):
    errors = []
    tokens = ["temperature", "average", "threshold", "delta", "xbar", "margin", "deviation", "reference"]

    for column in rows_df.columns:
        lowered = column.lower()
        if any(token in lowered for token in tokens):
            values = rows_df[column].dropna()
            if any(_has_more_than_four_decimals(value) for value in values):
                errors.append(f"{column} contains values with more than 4 decimal places")

    return errors


def _check_phase2_metrics(metrics_dict):
    errors = []
    for key in ["boundary_policy", "membership_policy", "synchronization_policy", "trigger_policy", "communication", "rows", "sensor_count"]:
        if key not in metrics_dict:
            errors.append(f"Missing metrics field: {key}")
    return errors


def _check_phase2_dictionary(dictionary_df, rows_df):
    errors = []
    expected_columns = ["field_name", "category", "measures", "calculation", "units"]
    missing_columns = [column for column in expected_columns if column not in dictionary_df.columns]
    if missing_columns:
        errors.append(f"phase2_data_dictionary.csv missing columns: {missing_columns}")

    if "field_name" not in dictionary_df.columns:
        return errors

    dictionary_fields = dictionary_df["field_name"].astype(str)
    if dictionary_fields.duplicated().any():
        duplicate_fields = sorted(set(dictionary_fields[dictionary_fields.duplicated()].tolist()))
        errors.append(
            "phase2_data_dictionary.csv has duplicate field definitions: "
            + ", ".join(duplicate_fields[:10])
        )

    row_fields = list(rows_df.columns)
    if len(dictionary_df) != len(row_fields):
        errors.append(
            "phase2_data_dictionary.csv row count does not match phase2_temperature_sensors.csv column count"
        )

    dictionary_field_set = set(dictionary_fields.tolist())
    row_field_set = set(row_fields)

    missing_from_dictionary = sorted(row_field_set - dictionary_field_set)
    extra_in_dictionary = sorted(dictionary_field_set - row_field_set)

    if missing_from_dictionary:
        errors.append(
            "phase2_data_dictionary.csv missing field definitions: "
            + ", ".join(missing_from_dictionary[:10])
        )
    if extra_in_dictionary:
        errors.append(
            "phase2_data_dictionary.csv has fields not present in phase2 rows: "
            + ", ".join(extra_in_dictionary[:10])
        )

    return errors


def _check_reduction_analysis_minimum_metrics(analysis_df):
    errors = []
    required_metrics = {
        "baseline_every_minute_messages",
        "poc_actual_messages",
        "reduction_difference_messages",
        "reduction_percent_primary",
    }

    if "metric" not in analysis_df.columns:
        errors.append("phase2_sensor_reduction_analysis.csv missing metric column")
        return errors

    observed = set(analysis_df["metric"].astype(str))
    missing = sorted(required_metrics - observed)
    if missing:
        errors.append(
            "phase2_sensor_reduction_analysis.csv missing required metrics: "
            + ", ".join(missing)
        )

    return errors


def _check_cross_artifact_consistency(rows_df, metrics_dict):
    errors = []

    metrics_rows = metrics_dict.get("rows")
    if isinstance(metrics_rows, (int, float)) and int(metrics_rows) != int(len(rows_df)):
        errors.append(
            f"metrics.rows ({metrics_rows}) does not match exported row count ({len(rows_df)})"
        )

    detected_sensor_count = len(_detect_sensor_names_from_rows(rows_df))
    metrics_sensor_count = metrics_dict.get("sensor_count")
    if isinstance(metrics_sensor_count, (int, float)) and int(metrics_sensor_count) != int(detected_sensor_count):
        errors.append(
            f"metrics.sensor_count ({metrics_sensor_count}) does not match detected exported sensors ({detected_sensor_count})"
        )

    communication = metrics_dict.get("communication")
    if isinstance(communication, dict):
        communication_pairs = [
            ("distributed_total_messages", "total_message_count"),
            ("distributed_trigger_messages", "trigger_message_count"),
            ("distributed_request_messages", "request_message_count"),
            ("distributed_response_messages", "response_message_count"),
            ("distributed_broadcast_messages", "broadcast_message_count"),
        ]
        for metrics_key, row_column in communication_pairs:
            if metrics_key in communication and row_column in rows_df.columns:
                row_sum = int(pd.to_numeric(rows_df[row_column], errors="coerce").fillna(0).sum())
                metrics_value = int(communication[metrics_key])
                if row_sum != metrics_value:
                    errors.append(
                        f"communication mismatch for {metrics_key}: metrics={metrics_value}, rows_sum={row_sum}"
                    )

    return errors


aligned_exists = PHASE1_ALIGNED_PATH.exists()
stats_exists = PHASE1_STATS_PATH.exists()
phase2_exists = PHASE2_OUTPUT_PATH.exists()
metrics_exists = PHASE2_METRICS_PATH.exists()
dictionary_exists = PHASE2_DICTIONARY_PATH.exists()
analysis_exists = PHASE2_ANALYSIS_PATH.exists()

print("aligned_exists", aligned_exists)
print("stats_exists", stats_exists)
print("phase2_exists", phase2_exists)
print("metrics_exists", metrics_exists)
print("dictionary_exists", dictionary_exists)
print("analysis_exists", analysis_exists)

if not aligned_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE1_ALIGNED_PATH}")
if not stats_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE1_STATS_PATH}")
if not phase2_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_OUTPUT_PATH}")
if not metrics_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_METRICS_PATH}")
if not dictionary_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_DICTIONARY_PATH}")
if not analysis_exists:
    raise FileNotFoundError(f"Missing artifact: {PHASE2_ANALYSIS_PATH}")

aligned_sample = pd.read_csv(PHASE1_ALIGNED_PATH, nrows=5)
sensor_columns = [
    column
    for column in aligned_sample.columns
    if column.endswith("_temperature")
    and column != "average_temperature_all_devices"
]
print("sample_rows", len(aligned_sample))
print("sensor_cols", len(sensor_columns))

rows_df = pd.read_csv(PHASE2_OUTPUT_PATH)
metrics_on_disk = json.loads(PHASE2_METRICS_PATH.read_text(encoding="utf-8"))
dictionary_df = pd.read_csv(PHASE2_DICTIONARY_PATH)
analysis_df = pd.read_csv(PHASE2_ANALYSIS_PATH)

validation_errors = []
validation_errors.extend(_check_phase2_schema(rows_df, analysis_df))
validation_errors.extend(_check_phase2_rounding(rows_df))
validation_errors.extend(_check_phase2_metrics(metrics_on_disk))
validation_errors.extend(_check_phase2_dictionary(dictionary_df, rows_df))
validation_errors.extend(_check_reduction_analysis_minimum_metrics(analysis_df))
validation_errors.extend(_check_cross_artifact_consistency(rows_df, metrics_on_disk))
validation_errors.extend(check_trace_invariants(rows_df, phase1_df, sensor_cols, p90_threshold))
evaluable_count = int(rows_df["global_state_evaluable"].sum())
if metrics_on_disk["evaluable_rows"] != evaluable_count:
    validation_errors.append("Evaluable row count does not reconcile")
if sum(metrics_on_disk["confusion"].values()) != evaluable_count:
    validation_errors.append("Confusion counts must exclude undefined global rows")
if metrics_on_disk["undefined_global_rows"] != len(rows_df) - evaluable_count:
    validation_errors.append("Undefined row count does not reconcile")
if metrics_on_disk["communication"]["centralized_observed_sensor_messages"] != int(rows_df["observed_online_sensor_count"].sum()):
    validation_errors.append("Observed-only centralized baseline does not reconcile")

if validation_errors:
    print("Phase 2 notebook-integrated validation failed:")
    for error in validation_errors:
        print(f"- {error}")
    raise ValueError("Notebook-integrated Phase 2 validation failed")

print("Phase 2 notebook-integrated validation passed.")

## Stage 5: History, Normalization, and Greedy Partitions (US-3.2 to US-3.4)

This stage keeps one discounted history value per sensor. On each row it updates history from the synchronization reference active at row entry, then uses that history snapshot for a completed synchronization. New references apply on the next row. Missing readings leave accumulated history unchanged; returning sensors synchronize before history updates resume. Rows with no valid observations do not update history or complete synchronization.

At each completed synchronization, normalize only eligible sensor histories once and share that immutable snapshot across all 19 alpha values. For each alpha, retain the singleton partition, repeatedly merge the worst-quality cluster with the partner minimizing merged-cluster quality, and stop at two clusters. Select the earliest minimum-quality candidate. Exact quality ties for clusters/partners use lexicographic sorted string IDs. One- and two-sensor cases require no merges. Empty eligibility creates no event; all-zero histories retain only the explicitly marked singleton fallback without normalization or merging.

The state tracker and clustering implementation remain in this notebook. Synchronization rows are checked against the validated Phase 2 trace. The existing `artifacts/phase3_sensor_history_events.csv` stores normalization snapshots and separate alpha evaluations with candidate scores, merge partner scores, and selected assignments. The existing `artifacts/phase3_sensor_history_trace.csv` retains one history-audit row per sensor per bucket. This stage constructs partitions but does not yet activate them for clustered monitoring or calculate FMIC/SMIC allowances.


In [ ]:
"""Source-aligned per-sensor history state for the Phase 3 clustering pipeline."""

from __future__ import annotations

from dataclasses import dataclass
import math
from numbers import Real
from types import MappingProxyType
from typing import Iterable, Mapping


@dataclass(frozen=True)
class SensorHistoryStep:
    """Auditable result of processing one row of sensor readings."""

    row_number: int
    online_sensors: tuple[str, ...]
    departed_sensors: tuple[str, ...]
    returned_sensors: tuple[str, ...]
    updated_sensors: tuple[str, ...]
    accumulated_histories: Mapping[str, float]
    history_audit_records: tuple[Mapping[str, object], ...]
    references_used: Mapping[str, float]
    reference_sync_ids_used: Mapping[str, int]
    synchronization_completed: bool
    synchronization_causes: tuple[str, ...]
    synchronization_id: int | None
    clustering_history_snapshot: Mapping[str, float] | None
    synchronized_references: Mapping[str, float] | None
    synchronized_reference_sync_ids: Mapping[str, int] | None


@dataclass(frozen=True)
class HistoryNormalizationSnapshot:
    """One event-level normalization result shared by alpha candidates."""

    eligible_sensor_ids: tuple[str, ...]
    raw_histories: Mapping[str, float]
    history_magnitudes: Mapping[str, float]
    maximum_history_magnitude: float | None
    normalized_histories: Mapping[str, float] | None
    normalization_status: str
    partition_basis: str
    zero_history_fallback: bool
    fallback_reason: str | None
    fallback_partition: tuple[tuple[str, ...], ...] | None
    deferred_reason: str | None


@dataclass(frozen=True)
class AlphaClusteringInput:
    """Per-alpha handoff referencing the shared normalization snapshot."""

    clustering_event_id: int
    synchronization_id: int
    alpha: float
    normalization_snapshot: HistoryNormalizationSnapshot


PHASE3_ALPHA_VALUES = tuple(value / 100 for value in range(5, 100, 5))


def normalize_sensor_histories(
    raw_histories: Mapping[str, Real],
    eligible_sensor_ids: Iterable[str],
) -> HistoryNormalizationSnapshot:
    """Normalize eligible scalar histories without mutating persistent state."""
    if isinstance(eligible_sensor_ids, str):
        raise ValueError("Provide eligible sensor IDs as an iterable, not a string.")
    eligible_ids = tuple(eligible_sensor_ids)
    if any(not isinstance(sensor_id, str) or not sensor_id for sensor_id in eligible_ids):
        raise ValueError("Eligible sensor IDs must be non-empty strings.")
    if len(set(eligible_ids)) != len(eligible_ids):
        raise ValueError("Eligible sensor IDs must be unique.")

    if not eligible_ids:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=(),
            raw_histories=MappingProxyType({}),
            history_magnitudes=MappingProxyType({}),
            maximum_history_magnitude=None,
            normalized_histories=None,
            normalization_status="deferred",
            partition_basis="not_applicable",
            zero_history_fallback=False,
            fallback_reason=None,
            fallback_partition=None,
            deferred_reason="no_eligible_sensors",
        )

    missing = sorted(set(eligible_ids) - set(raw_histories))
    if missing:
        raise ValueError(f"Raw history is missing eligible sensors: {missing}.")

    eligible_histories: dict[str, float] = {}
    for sensor_id in sorted(eligible_ids):
        value = raw_histories[sensor_id]
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"History for {sensor_id} must be a finite number.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value):
            raise ValueError(f"History for {sensor_id} must be finite.")
        eligible_histories[sensor_id] = numeric_value

    magnitudes = {
        sensor_id: abs(history)
        for sensor_id, history in eligible_histories.items()
    }
    maximum_magnitude = max(magnitudes.values())
    if maximum_magnitude == 0:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=tuple(sorted(eligible_histories)),
            raw_histories=MappingProxyType(eligible_histories),
            history_magnitudes=MappingProxyType(magnitudes),
            maximum_history_magnitude=0.0,
            normalized_histories=None,
            normalization_status="all_zero_history_fallback",
            partition_basis="all_zero_history_fallback",
            zero_history_fallback=True,
            fallback_reason="all_eligible_histories_are_zero",
            fallback_partition=tuple(
                (sensor_id,) for sensor_id in sorted(eligible_histories)
            ),
            deferred_reason=None,
        )

    normalized = {
        sensor_id: history / maximum_magnitude
        for sensor_id, history in eligible_histories.items()
    }
    return HistoryNormalizationSnapshot(
        eligible_sensor_ids=tuple(sorted(eligible_histories)),
        raw_histories=MappingProxyType(eligible_histories),
        history_magnitudes=MappingProxyType(magnitudes),
        maximum_history_magnitude=maximum_magnitude,
        normalized_histories=MappingProxyType(normalized),
        normalization_status="normalized",
        partition_basis="normalized_history",
        zero_history_fallback=False,
        fallback_reason=None,
        fallback_partition=None,
        deferred_reason=None,
    )


def build_alpha_clustering_inputs(
    synchronization_id: int,
    normalization_snapshot: HistoryNormalizationSnapshot,
) -> tuple[AlphaClusteringInput, ...]:
    """Create the full alpha sweep, sharing one event normalization result."""
    if synchronization_id < 1:
        raise ValueError("Synchronization ID must be positive.")
    if normalization_snapshot.deferred_reason is not None:
        return ()
    return tuple(
        AlphaClusteringInput(
            clustering_event_id=synchronization_id,
            synchronization_id=synchronization_id,
            alpha=alpha,
            normalization_snapshot=normalization_snapshot,
        )
        for alpha in PHASE3_ALPHA_VALUES
    )


Cluster = tuple[str, ...]
Partition = tuple[Cluster, ...]


def _validated_clustering_values(
    normalized_histories: Mapping[str, Real], alpha: Real,
) -> dict[str, float]:
    if isinstance(alpha, bool) or not isinstance(alpha, Real):
        raise ValueError("Alpha must be a finite number in [0, 1].")
    if not math.isfinite(float(alpha)) or not 0 <= alpha <= 1:
        raise ValueError("Alpha must be a finite number in [0, 1].")
    values: dict[str, float] = {}
    for sensor_id, value in normalized_histories.items():
        if not isinstance(sensor_id, str) or not sensor_id:
            raise ValueError("Clustering sensor IDs must be non-empty strings.")
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"Normalized history for {sensor_id} must be finite.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value) or abs(numeric_value) > 1:
            raise ValueError(f"Normalized history for {sensor_id} must be in [-1, 1].")
        values[sensor_id] = numeric_value
    return values


def _canonical_cluster(cluster: Iterable[str]) -> Cluster:
    if isinstance(cluster, str):
        raise ValueError("A cluster must be an iterable of sensor IDs, not a string.")
    members = tuple(cluster)
    if not members or any(not isinstance(member, str) or not member for member in members):
        raise ValueError("Clusters must contain non-empty string sensor IDs.")
    if len(set(members)) != len(members):
        raise ValueError("A cluster must not contain duplicate sensor IDs.")
    return tuple(sorted(members))


def _cluster_quality(cluster: Cluster, values: Mapping[str, float], alpha: float) -> float:
    return abs(math.fsum(values[sensor_id] for sensor_id in cluster)) / len(cluster) + alpha * len(cluster)


def cluster_quality(
    cluster: Iterable[str], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score signed cancellation before applying the scalar absolute value."""
    values = _validated_clustering_values(normalized_histories, alpha)
    members = _canonical_cluster(cluster)
    unknown = sorted(set(members) - set(values))
    if unknown:
        raise ValueError(f"Cluster contains sensors without normalized histories: {unknown}.")
    return _cluster_quality(members, values, float(alpha))


def partition_quality(
    partition: Iterable[Iterable[str]], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score a disjoint partition containing exactly the supplied sensors."""
    values = _validated_clustering_values(normalized_histories, alpha)
    clusters = tuple(sorted(_canonical_cluster(cluster) for cluster in partition))
    members = tuple(member for cluster in clusters for member in cluster)
    if not clusters or len(set(members)) != len(members) or set(members) != set(values):
        raise ValueError("Partition must cover every eligible sensor exactly once.")
    return max(_cluster_quality(cluster, values, float(alpha)) for cluster in clusters)


def evaluate_alpha_clustering(clustering_input: AlphaClusteringInput) -> dict[str, object] | None:
    """Retain the greedy candidate sequence and choose its earliest minimum."""
    snapshot = clustering_input.normalization_snapshot
    if snapshot.deferred_reason is not None:
        return None
    eligible = snapshot.eligible_sensor_ids
    if not eligible:
        raise ValueError("A completed clustering input must have eligible sensors.")
    if snapshot.zero_history_fallback:
        if any(snapshot.raw_histories[sensor_id] != 0 for sensor_id in eligible):
            raise ValueError("All-zero fallback requires zero raw eligible histories.")
        # Zero raw histories allow fallback scoring without fabricating normalization.
        source_values = snapshot.raw_histories
    else:
        if snapshot.normalized_histories is None:
            raise ValueError("Nonzero history clustering requires normalized histories.")
        source_values = snapshot.normalized_histories
    values = _validated_clustering_values(source_values, clustering_input.alpha)
    if len(set(eligible)) != len(eligible) or set(values) != set(eligible):
        raise ValueError("Clustering values must cover exactly the eligible sensor set.")
    alpha = float(clustering_input.alpha)
    partition: Partition = tuple((sensor_id,) for sensor_id in sorted(eligible))
    candidates: list[dict[str, object]] = []
    candidate_qualities: list[float] = []
    candidate_partitions: list[Partition] = []
    merges: list[dict[str, object]] = []

    while True:
        qualities = {cluster: _cluster_quality(cluster, values, alpha) for cluster in partition}
        quality = max(qualities.values())
        candidates.append({
            "candidate_index": len(candidates),
            "partition": partition,
            "cluster_count": len(partition),
            "cluster_qualities": [
                {"members": cluster, "quality": qualities[cluster]} for cluster in partition
            ],
            "partition_quality": quality,
        })
        candidate_qualities.append(quality)
        candidate_partitions.append(partition)
        if snapshot.zero_history_fallback or len(partition) <= 2:
            break

        worst = min(partition, key=lambda cluster: (-qualities[cluster], cluster))
        partners = []
        for partner in partition:
            if partner != worst:
                merged = tuple(sorted(worst + partner))
                partners.append((partner, merged, _cluster_quality(merged, values, alpha)))
        partner, merged, merged_quality = min(partners, key=lambda item: (item[2], item[0]))
        merges.append({
            "from_candidate_index": len(candidates) - 1,
            "worst_cluster": worst,
            "worst_cluster_quality": qualities[worst],
            "partner_scores": [
                {"partner": item[0], "merged_cluster": item[1], "merged_quality": item[2]}
                for item in partners
            ],
            "selected_partner": partner,
            "merged_cluster": merged,
            "merged_quality": merged_quality,
        })
        partition = tuple(sorted(
            tuple(cluster for cluster in partition if cluster not in (worst, partner)) + (merged,)
        ))

    selected_index = min(range(len(candidates)), key=lambda index: candidate_qualities[index])
    selected_partition = candidate_partitions[selected_index]
    return {
        "clustering_event_id": clustering_input.clustering_event_id,
        "synchronization_id": clustering_input.synchronization_id,
        "normalization_snapshot_id": clustering_input.synchronization_id,
        "alpha": alpha,
        "partition_basis": snapshot.partition_basis,
        "zero_history_fallback": snapshot.zero_history_fallback,
        "fallback_reason": snapshot.fallback_reason,
        "candidate_partitions": candidates,
        "merge_steps": merges,
        "selected_candidate_index": selected_index,
        "selected_partition": selected_partition,
        "partition_quality": candidate_qualities[selected_index],
        "cluster_count": len(selected_partition),
        "cluster_assignments": {
            sensor_id: cluster_index
            for cluster_index, cluster in enumerate(selected_partition)
            for sensor_id in cluster
        },
    }


class SensorHistoryTracker:
    """Maintain discounted sensor histories and synchronization references.

    History updates use the active reference on row entry. Synchronization
    snapshots the completed interval and installs new references for the next
    row without resetting accumulated history.
    """

    DISCOUNT = 0.5

    def __init__(self, sensor_ids: Iterable[str]) -> None:
        if isinstance(sensor_ids, str):
            raise ValueError("Provide sensor IDs as an iterable, not a single string.")
        ids = tuple(sensor_ids)
        if not ids:
            raise ValueError("At least one sensor ID is required.")
        if any(not isinstance(sensor_id, str) or not sensor_id for sensor_id in ids):
            raise ValueError("Sensor IDs must be non-empty strings.")
        if len(set(ids)) != len(ids):
            raise ValueError("Sensor IDs must be unique.")

        self._sensor_ids = tuple(sorted(ids))
        self._histories = {sensor_id: 0.0 for sensor_id in self._sensor_ids}
        self._references: dict[str, float | None] = {
            sensor_id: None for sensor_id in self._sensor_ids
        }
        self._reference_sync_ids: dict[str, int | None] = {
            sensor_id: None for sensor_id in self._sensor_ids
        }
        self._previous_online: set[str] = set()
        self._initialized = False
        self._pending_causes: set[str] = set()
        self._row_number = 0
        self._next_sync_id = 1

    @property
    def histories(self) -> dict[str, float]:
        """Return a copy of accumulated per-sensor history values."""
        return dict(self._histories)

    def step(
        self,
        readings: Mapping[str, Real | None],
        *,
        synchronization_causes: Iterable[str] = (),
    ) -> SensorHistoryStep:
        """Process one bucket and optionally complete a synchronization.

        Every configured sensor must be present in ``readings``. Use ``None``
        or NaN for a missing reading. Membership changes request synchronization
        automatically; other causes are supplied by the monitoring engine.
        """
        reading_keys = set(readings)
        expected_keys = set(self._sensor_ids)
        if reading_keys != expected_keys:
            missing = sorted(expected_keys - reading_keys)
            extra = sorted(reading_keys - expected_keys)
            raise ValueError(
                f"Reading sensor IDs do not match tracker; missing={missing}, extra={extra}."
            )

        observed: dict[str, float | None] = {}
        for sensor_id in self._sensor_ids:
            value = readings[sensor_id]
            if value is None:
                observed[sensor_id] = None
                continue
            if isinstance(value, bool) or not isinstance(value, Real):
                raise ValueError(f"Reading for {sensor_id} must be a finite number or missing.")
            numeric_value = float(value)
            if math.isnan(numeric_value):
                observed[sensor_id] = None
            elif not math.isfinite(numeric_value):
                raise ValueError(f"Reading for {sensor_id} must be finite.")
            else:
                observed[sensor_id] = numeric_value

        if isinstance(synchronization_causes, str):
            raise ValueError("Provide synchronization causes as an iterable of strings.")
        explicit_causes = set()
        for cause in synchronization_causes:
            if not isinstance(cause, str) or not cause.strip():
                raise ValueError("Synchronization causes must be non-empty strings.")
            explicit_causes.add(cause.strip())

        online = {sensor_id for sensor_id, value in observed.items() if value is not None}
        departed = self._previous_online - online if self._initialized else set()
        returned = online - self._previous_online if self._initialized else set()
        causes = set(self._pending_causes) | explicit_causes
        if not self._initialized:
            causes.add("initialization")
        if departed or returned:
            causes.add("sensor_membership_changed")

        references_used: dict[str, float] = {}
        reference_sync_ids_used: dict[str, int] = {}
        updated_sensors = []
        row_audit_records = []
        for sensor_id in self._sensor_ids:
            value = observed[sensor_id]
            pre_history = self._histories[sensor_id]
            entry_reference = self._references[sensor_id]
            entry_sync_id = self._reference_sync_ids[sensor_id]
            signed_deviation: float | None = None
            history_updated = False

            if value is None:
                update_reason = "missing_observation"
            elif entry_reference is None:
                update_reason = (
                    "awaiting_reentry_synchronization"
                    if sensor_id in returned
                    else "awaiting_initial_reference"
                )
            else:
                if entry_sync_id is None:
                    raise RuntimeError(
                        f"Sensor {sensor_id} has a reference without a synchronization ID."
                    )
                signed_deviation = value - entry_reference
                references_used[sensor_id] = entry_reference
                reference_sync_ids_used[sensor_id] = entry_sync_id
                self._histories[sensor_id] = (
                    self.DISCOUNT * pre_history + signed_deviation
                )
                history_updated = True
                updated_sensors.append(sensor_id)
                update_reason = "updated"

            row_audit_records.append({
                "row_number": self._row_number,
                "sensor_id": sensor_id,
                "observed_value": value,
                "active_reference": entry_reference,
                "reference_sync_id": entry_sync_id,
                "history_before": pre_history,
                "signed_deviation": signed_deviation,
                "history_after": self._histories[sensor_id],
                "history_update_performed": int(history_updated),
                "history_update_reason": update_reason,
                "synchronization_id": None,
            })

        synchronization_completed = bool(online and causes)
        synchronization_id = None
        clustering_history_snapshot: dict[str, float] | None = None
        synchronized_references: dict[str, float] | None = None
        synchronized_reference_sync_ids: dict[str, int] | None = None
        if synchronization_completed:
            synchronization_id = self._next_sync_id
            self._next_sync_id += 1
            clustering_history_snapshot = {
                sensor_id: self._histories[sensor_id]
                for sensor_id in sorted(online)
            }
            synchronized_references = {
                sensor_id: float(observed[sensor_id])
                for sensor_id in sorted(online)
                if observed[sensor_id] is not None
            }
            synchronized_reference_sync_ids = {
                sensor_id: synchronization_id for sensor_id in synchronized_references
            }
            for sensor_id, reference in synchronized_references.items():
                self._references[sensor_id] = reference
                self._reference_sync_ids[sensor_id] = synchronization_id
            self._initialized = True
            self._pending_causes.clear()
        else:
            self._pending_causes = causes

        for sensor_id in set(self._sensor_ids) - online:
            self._references[sensor_id] = None
            self._reference_sync_ids[sensor_id] = None

        for audit_record in row_audit_records:
            audit_record["synchronization_id"] = synchronization_id

        self._previous_online = online
        result = SensorHistoryStep(
            row_number=self._row_number,
            online_sensors=tuple(sorted(online)),
            departed_sensors=tuple(sorted(departed)),
            returned_sensors=tuple(sorted(returned)),
            updated_sensors=tuple(updated_sensors),
            accumulated_histories=dict(self._histories),
            history_audit_records=tuple(row_audit_records),
            references_used=references_used,
            reference_sync_ids_used=reference_sync_ids_used,
            synchronization_completed=synchronization_completed,
            synchronization_causes=tuple(sorted(causes)),
            synchronization_id=synchronization_id,
            clustering_history_snapshot=clustering_history_snapshot,
            synchronized_references=synchronized_references,
            synchronized_reference_sync_ids=synchronized_reference_sync_ids,
        )
        self._row_number += 1
        return result


In [ ]:
# Stage 5: Build history, normalize once per event, and evaluate all alpha partitions.
if len(phase1_df) != len(phase2_df):
    raise ValueError(
        "Phase 3 history input and matched Phase 2 trace have different row counts."
    )

history_tracker = SensorHistoryTracker(sensor_cols)
phase3_history_event_records = []
phase3_history_audit_records = []
phase3_alpha_clustering_inputs = []
for row_index in range(len(phase1_df)):
    input_row = phase1_df.iloc[row_index]
    baseline_row = phase2_df.iloc[row_index]
    if input_row["bucket_epoch"] != baseline_row["bucket_epoch"]:
        raise ValueError(
            f"Phase 3 history input and matched Phase 2 trace have different "
            f"bucket epochs at row {row_index}."
        )
    causes = ()
    if int(baseline_row["event_resync_performed"]) == 1:
        causes = tuple(
            cause
            for cause in str(baseline_row["event_resync_causes"]).split("|")
            if cause
        )
    history_step = history_tracker.step(
        {column: input_row[column] for column in sensor_cols},
        synchronization_causes=causes,
    )
    baseline_sync = int(baseline_row["event_resync_performed"]) == 1
    if history_step.synchronization_completed != baseline_sync:
        bucket = input_row["bucket_epoch"]
        raise ValueError(
            f"Phase 3 history synchronization mismatch at bucket {bucket}: "
            f"history={history_step.synchronization_completed}, Phase 2={baseline_sync}."
        )

    for audit_record in history_step.history_audit_records:
        phase3_history_audit_records.append({
            "bucket_epoch": int(input_row["bucket_epoch"]),
            "bucket_time_utc": input_row["bucket_time_utc"],
            **audit_record,
        })

    if not history_step.synchronization_completed:
        continue
    if history_step.synchronization_id is None:
        raise RuntimeError("Completed synchronization has no synchronization ID.")
    normalization_snapshot = normalize_sensor_histories(
        history_step.accumulated_histories, history_step.online_sensors
    )
    event_alpha_inputs = build_alpha_clustering_inputs(
        history_step.synchronization_id, normalization_snapshot
    )
    phase3_alpha_clustering_inputs.extend(event_alpha_inputs)
    alpha_evaluations = [evaluate_alpha_clustering(item) for item in event_alpha_inputs]
    fallback_partition = normalization_snapshot.fallback_partition
    phase3_history_event_records.append({
        "bucket_epoch": int(input_row["bucket_epoch"]),
        "bucket_time_utc": input_row["bucket_time_utc"],
        "history_row_number": history_step.row_number,
        "synchronization_id": history_step.synchronization_id,
        "clustering_event_id": history_step.synchronization_id,
        "normalization_snapshot_id": history_step.synchronization_id,
        "normalization_status": normalization_snapshot.normalization_status,
        "eligible_sensor_ids": "|".join(
            normalization_snapshot.eligible_sensor_ids
        ),
        "eligible_raw_histories": json.dumps(
            dict(normalization_snapshot.raw_histories), sort_keys=True
        ),
        "eligible_history_magnitudes": json.dumps(
            dict(normalization_snapshot.history_magnitudes), sort_keys=True
        ),
        "maximum_history_magnitude": normalization_snapshot.maximum_history_magnitude,
        "normalized_histories": json.dumps(
            (dict(normalization_snapshot.normalized_histories)
             if normalization_snapshot.normalized_histories is not None else None),
            sort_keys=True
        ),
        "partition_basis": normalization_snapshot.partition_basis,
        "zero_history_fallback": int(
            normalization_snapshot.zero_history_fallback
        ),
        "fallback_reason": normalization_snapshot.fallback_reason,
        "fallback_partition": json.dumps(fallback_partition),
        "alpha_clustering_evaluations": json.dumps(alpha_evaluations, sort_keys=True, allow_nan=False),
        "synchronization_causes": "|".join(history_step.synchronization_causes),
        "online_sensor_ids": "|".join(history_step.online_sensors),
        "history_updated_sensor_ids": "|".join(history_step.updated_sensors),
        "history_references_used": json.dumps(
            history_step.references_used, sort_keys=True
        ),
        "reference_sync_ids_used": json.dumps(
            history_step.reference_sync_ids_used, sort_keys=True
        ),
        "clustering_history_snapshot": json.dumps(
            history_step.clustering_history_snapshot, sort_keys=True
        ),
        "synchronized_references": json.dumps(
            history_step.synchronized_references, sort_keys=True
        ),
        "synchronized_reference_sync_ids": json.dumps(
            history_step.synchronized_reference_sync_ids, sort_keys=True
        ),
    })

phase3_history_events = pd.DataFrame.from_records(
    phase3_history_event_records,
    columns=[
        "bucket_epoch", "bucket_time_utc", "history_row_number",
        "synchronization_id", "clustering_event_id",
        "normalization_snapshot_id", "normalization_status",
        "synchronization_causes", "online_sensor_ids",
        "history_updated_sensor_ids", "history_references_used",
        "reference_sync_ids_used", "clustering_history_snapshot",
        "synchronized_references", "synchronized_reference_sync_ids",
        "eligible_sensor_ids", "eligible_raw_histories",
        "eligible_history_magnitudes", "maximum_history_magnitude",
        "normalized_histories", "partition_basis",
        "zero_history_fallback", "fallback_reason", "fallback_partition",
        "alpha_clustering_evaluations",
    ],
)
phase3_history_audit = pd.DataFrame.from_records(
    phase3_history_audit_records,
    columns=[
        "bucket_epoch", "bucket_time_utc", "row_number", "sensor_id",
        "observed_value", "active_reference", "reference_sync_id",
        "history_before", "signed_deviation", "history_after",
        "history_update_performed", "history_update_reason",
        "synchronization_id",
    ],
)
PHASE3_HISTORY_EVENTS_PATH = REPO_ROOT / "artifacts" / "phase3_sensor_history_events.csv"
PHASE3_HISTORY_AUDIT_PATH = REPO_ROOT / "artifacts" / "phase3_sensor_history_trace.csv"
PHASE3_HISTORY_EVENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
phase3_history_events.to_csv(PHASE3_HISTORY_EVENTS_PATH, index=False)
phase3_history_audit.to_csv(PHASE3_HISTORY_AUDIT_PATH, index=False)
print(
    f"Saved {len(phase3_history_events)} Phase 3 event snapshots to "
    f"{PHASE3_HISTORY_EVENTS_PATH} and "
    f"{len(phase3_history_audit)} per-sensor audit rows to "
    f"{PHASE3_HISTORY_AUDIT_PATH}"
)
